In [1]:
import numpy as np

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics.pairwise import cosine_similarity

In [2]:

DATA_PATH = "book_bestseller_clean.csv"

df_books = pd.read_csv(

    DATA_PATH,

    encoding="utf-8-sig",

)

print("데이터 크기:", df_books.shape)

print("컬럼:", df_books.columns.tolist())

print("상품명 결측치:", df_books["상품명"].isna().sum())

df_books[["상품명"]].head(10)

데이터 크기: (199, 8)
컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']
상품명 결측치: 0


,상품명
0,소년이 온다
1,모순
2,결국 국민이 합니다
3,혼모노
4,급류
5,초역 부처의 말
6,청춘의 독서(특별증보판)
7,어른의 행복은 조용하다
8,채식주의자
9,단 한 번의 삶(강물에디션 활판인쇄 한정판)


In [3]:

df_reco = df_books.copy()

df_reco["상품명"] = (

    df_reco["상품명"]

    .fillna("")

    .astype(str)

    .str.strip()

)

df_reco = (

    df_reco[df_reco["상품명"] != ""]

    .reset_index(drop=True)

)

print("추천에 사용할 도서 수:", len(df_reco))

추천에 사용할 도서 수: 199


In [5]:
print("상품명 결측치:", df_reco["상품명"].isna().sum())

상품명 결측치: 0


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 도서 제목 가져오기
titles = df_reco["상품명"]

# TF-IDF Vectorizer 생성
tfidf = TfidfVectorizer()

# 도서 제목을 TF-IDF 벡터로 변환
tfidf_matrix = tfidf.fit_transform(titles)

# 결과 확인
print("도서 수:", tfidf_matrix.shape[0])
print("단어 수:", tfidf_matrix.shape[1])
print("행렬 크기:", tfidf_matrix.shape)

도서 수: 199
단어 수: 536
행렬 크기: (199, 536)


In [6]:

selected_index = 0

selected_title = df_reco.loc[

    selected_index,

    "상품명",

]

print("선택 도서:", selected_title)

선택 도서: 소년이 온다


In [7]:

from sklearn.metrics.pairwise import cosine_similarity

# 선택한 도서의 TF-IDF 벡터 가져오기
selected_vector = tfidf_matrix[selected_index]

# 선택 도서와 전체 도서의 코사인 유사도 계산
similarity_scores = cosine_similarity(
    selected_vector,
    tfidf_matrix,
).flatten()

# 결과 확인
print("유사도 개수:", len(similarity_scores))
print("전체 도서 수:", len(df_reco))


유사도 개수: 199
전체 도서 수: 199


In [8]:
import numpy as np
import pandas as pd

# 도서별 유사도 결과 DataFrame 생성
score_df = pd.DataFrame({
    "index": np.arange(len(df_reco)),
    "상품명": df_reco["상품명"].to_numpy(),
    "similarity": similarity_scores,
})

# 유사도가 높은 순서대로 상위 10개 확인
score_df.sort_values(
    "similarity",
    ascending=False,
).head(10)

,index,상품명,similarity
0,0,소년이 온다,1.0
1,1,모순,0.0
2,2,결국 국민이 합니다,0.0
3,3,혼모노,0.0
4,4,급류,0.0
5,5,초역 부처의 말,0.0
6,6,청춘의 독서(특별증보판),0.0
7,7,어른의 행복은 조용하다,0.0
8,8,채식주의자,0.0
9,9,단 한 번의 삶(강물에디션 활판인쇄 한정판),0.0
